# Umi-OCR Basic Usage in Jupyter

This notebook demonstrates how to use Umi-OCR's HTTP API from Python/Jupyter to perform OCR on images.

## Prerequisites

1. Umi-OCR must be running on your system
2. HTTP service must be enabled in Umi-OCR settings (enabled by default)
3. Default port is `1224` (can be changed in global settings)

## Installation

Install required Python packages:

In [ ]:
# Install required packages
!pip install requests pillow

## Import Libraries

In [ ]:
import requests
import base64
import json
from pathlib import Path
from PIL import Image
import io

## Configuration

In [ ]:
# Umi-OCR HTTP API configuration
UMI_OCR_HOST = "127.0.0.1"
UMI_OCR_PORT = 1224
BASE_URL = f"http://{UMI_OCR_HOST}:{UMI_OCR_PORT}"

## Helper Functions

In [ ]:
def image_to_base64(image_path):
    """Convert an image file to base64 string."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def pil_image_to_base64(pil_image):
    """Convert a PIL Image to base64 string."""
    buffer = io.BytesIO()
    pil_image.save(buffer, format='PNG')
    return base64.b64encode(buffer.getvalue()).decode('utf-8')

## 1. Check OCR Options

First, let's query the available OCR parameters:

In [ ]:
def get_ocr_options():
    """Get available OCR options and parameters."""
    url = f"{BASE_URL}/api/ocr/get_options"
    response = requests.get(url)
    return response.json()

# Get and display OCR options
options = get_ocr_options()
print("Available OCR Options:")
print(json.dumps(options, indent=2, ensure_ascii=False))

## 2. Perform OCR on an Image

Now let's perform OCR on an image file:

In [ ]:
def ocr_image(image_base64, options=None):
    """Perform OCR on a base64 encoded image.
    
    Args:
        image_base64: Base64 encoded image string
        options: Dictionary of OCR options (optional)
        
    Returns:
        Dictionary containing OCR results
    """
    url = f"{BASE_URL}/api/ocr"
    
    payload = {
        "base64": image_base64
    }
    
    # Add custom options if provided
    if options:
        payload.update(options)
    
    response = requests.post(url, json=payload)
    return response.json()

# Example: OCR an image file
# Replace 'your_image.png' with your actual image path
image_path = "your_image.png"

# Check if file exists
if Path(image_path).exists():
    # Convert image to base64
    image_b64 = image_to_base64(image_path)
    
    # Perform OCR
    result = ocr_image(image_b64)
    
    # Display results
    print("OCR Result:")
    print(json.dumps(result, indent=2, ensure_ascii=False))
    
    # Extract and display text only
    if result.get('code') == 100:
        print("\nExtracted Text:")
        for item in result.get('data', []):
            print(item.get('text', ''))
else:
    print(f"Image file '{image_path}' not found. Please provide a valid image path.")

## 3. OCR with Custom Options

You can customize OCR behavior by passing options:

In [ ]:
# Example with custom options
if Path(image_path).exists():
    image_b64 = image_to_base64(image_path)
    
    # Custom options
    custom_options = {
        "ocr.language": "models/config_chinese.txt",  # Chinese language
        "ocr.cls": True,  # Enable text direction correction
        "tbpu.parser": "multi_para"  # Multi-column paragraph parsing
    }
    
    result = ocr_image(image_b64, custom_options)
    
    print("OCR Result with Custom Options:")
    if result.get('code') == 100:
        print("\nExtracted Text:")
        for item in result.get('data', []):
            print(item.get('text', ''))
    else:
        print(f"Error: {result.get('data', 'Unknown error')}")
else:
    print(f"Image file '{image_path}' not found.")

## 4. Working with PIL Images

You can also work directly with PIL Image objects:

In [ ]:
# Create a simple test image with text
from PIL import Image, ImageDraw, ImageFont

# Create a white image
img = Image.new('RGB', (400, 100), color='white')
draw = ImageDraw.Draw(img)

# Draw some text
draw.text((10, 30), "Hello from Umi-OCR!", fill='black')

# Display the image
display(img)

# Convert to base64 and perform OCR
img_b64 = pil_image_to_base64(img)
result = ocr_image(img_b64)

print("\nOCR Result:")
if result.get('code') == 100:
    for item in result.get('data', []):
        print(f"Text: {item.get('text', '')}")
        print(f"Confidence: {item.get('score', 0):.2f}")
else:
    print(f"Error: {result.get('data', 'Unknown error')}")

## 5. Error Handling

Always check the response code and handle errors appropriately:

In [ ]:
def ocr_image_safe(image_base64, options=None):
    """Perform OCR with error handling."""
    try:
        result = ocr_image(image_base64, options)
        
        # Check response code
        code = result.get('code')
        if code == 100:
            # Success
            return {'success': True, 'data': result.get('data', [])}
        elif code == 101:
            # No text detected
            return {'success': True, 'data': [], 'message': 'No text detected in image'}
        else:
            # Error occurred
            return {'success': False, 'error': result.get('data', 'Unknown error')}
    except requests.exceptions.ConnectionError:
        return {'success': False, 'error': 'Cannot connect to Umi-OCR. Make sure it is running.'}
    except Exception as e:
        return {'success': False, 'error': str(e)}

# Test with error handling
if Path(image_path).exists():
    image_b64 = image_to_base64(image_path)
    result = ocr_image_safe(image_b64)
    
    if result['success']:
        print("OCR succeeded!")
        for item in result['data']:
            print(item.get('text', ''))
    else:
        print(f"OCR failed: {result['error']}")
else:
    print(f"Please provide a valid image path to test OCR.")

## Summary

This notebook demonstrated:
- How to query available OCR options
- How to perform basic OCR on images
- How to use custom OCR options
- How to work with PIL images
- How to handle errors properly

For more advanced usage, see:
- `02_batch_processing.ipynb` - Batch processing multiple images
- `03_qrcode_operations.ipynb` - QR code detection and generation

## References

- [Umi-OCR GitHub](https://github.com/hiroi-sora/Umi-OCR)
- [HTTP API Documentation](../../docs/http/README.md)
- [Command Line Documentation](../../docs/README_CLI.md)